# CS 435 — Project 3: Medical X-Ray Anomaly Detection
## Phase 3: Conditional GAN for Minority-Class X-Ray Synthesis

**Goal of this notebook:**  
Train a Generative Adversarial Network (GAN) that learns to synthesize realistic chest X-ray images for the minority disease classes identified in Phase 1 (`Pneumonia` and `Cardiomegaly`). The synthetic images produced here will be added to the training set in Phase 4, then we retrain the CNN and compare results against the Phase 2 baseline.

### How a GAN works (in plain English)
A GAN pits two neural networks against each other in a competition:
- **Generator (G):** Takes a random noise vector as input and tries to produce a fake X-ray that looks real
- **Discriminator (D):** Looks at an image and tries to tell whether it is a real X-ray from the dataset or a fake one produced by G

They train simultaneously: G gets better at fooling D, and D gets better at catching G. As Brownlee describes in *GANs with Python*, at convergence the generator produces images indistinguishable from real ones and the discriminator is left guessing at 50/50.

### Why a conditional GAN?
A standard GAN generates random images from the learned distribution. A **conditional GAN (cGAN)** accepts a class label as extra input to both G and D, so we can say "generate a Pneumonia X-ray" specifically. This is exactly what we need to oversample the minority classes.

**Prerequisites:** Phase 1 splits must be saved to Google Drive.

---
## Step 1: Mount Drive and import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## Step 2: Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.abspath("")
SPLITS_DIR  = os.path.join(BASE_DIR, 'splits')
GAN_DIR     = os.path.join(BASE_DIR, 'gan_models')
SYNTH_DIR   = os.path.join(BASE_DIR, 'synthetic_images')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
for d in [GAN_DIR, SYNTH_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Image settings (must match Phase 1 & 2) ───────────────────────────────────
IMG_SIZE     = 64
CHANNELS     = 1
LATENT_DIM   = 100

ALL_CLASSES      = ['No Finding', 'Atelectasis', 'Effusion', 'Cardiomegaly', 'Pneumonia']
MINORITY_CLASSES = ['Pneumonia', 'Cardiomegaly']

BATCH_SIZE           = 32
GAN_EPOCHS           = 200
SAMPLE_INTERVAL      = 20
N_SYNTHETIC_PER_CLASS = 2000

print("Configuration loaded.")
print(f"  GAN image size      : {IMG_SIZE}×{IMG_SIZE}")
print(f"  Latent dim          : {LATENT_DIM}")
print(f"  Minority classes    : {MINORITY_CLASSES}")
print(f"  Synthetic per class : {N_SYNTHETIC_PER_CLASS}")

---
## Step 3: Load and prepare training data for the GAN

The GAN only trains on images from the **training split** (never validation or test — that would contaminate evaluation). We load the train CSV from Phase 1 and extract images per minority class.

### Important: GAN normalization range is [-1, 1], not [0, 1]
The generator output layer uses `tanh` activation, which outputs values in [-1, 1]. So we must normalize input images to the same range: `pixel / 127.5 - 1.0`. This is standard DCGAN practice as described by Brownlee (*GANs with Python*, Ch. 5 and 10).

In [ ]:
def load_class_images(dataframe, class_name, img_size=64):
    """
    Load all images for a specific disease class from disk.
    Returns a float32 numpy array normalized to [-1, 1].

    Args:
        dataframe  : train DataFrame from Phase 1
        class_name : one of the TARGET_CLASSES strings
        img_size   : target image size (default 64 for GAN)

    Returns:
        np.ndarray of shape (N, img_size, img_size, 1), values in [-1, 1]
    """
    # Filter to rows where this class label is positive (=1)
    class_df = dataframe[dataframe[class_name] == 1].reset_index(drop=True)
    print(f"  Loading {len(class_df)} '{class_name}' images...")

    images = []
    for filepath in class_df['filepath']:
        # Read and decode the image
        raw   = tf.io.read_file(filepath)
        img   = tf.image.decode_png(raw, channels=CHANNELS)
        # Resize to GAN training size (64×64)
        img   = tf.image.resize(img, [img_size, img_size])
        # Normalize from [0, 255] → [-1, 1] — required for tanh output layer
        img   = (tf.cast(img, tf.float32) - 127.5) / 127.5
        images.append(img.numpy())

    images = np.array(images)   # shape: (N, 64, 64, 1)
    print(f"  Done. Array shape: {images.shape}, value range: [{images.min():.2f}, {images.max():.2f}]")
    return images


# Load the training split
train_df = pd.read_csv(os.path.join(SPLITS_DIR, 'train.csv'))
print(f"Train split loaded: {len(train_df):,} images")
print("\nClass counts in training split:")
print(train_df[ALL_CLASSES].sum().sort_values(ascending=False))

---
## Step 4: Build the Generator

The generator takes a random noise vector (shape: `(LATENT_DIM,)`) and produces a 64×64 grayscale X-ray image.

### Architecture (Dense → Reshape → 4× Conv2DTranspose):
1. **Dense layer** — expands the 100-dim noise vector into enough activations to reshape into a small spatial feature map (4×4×256)
2. **Reshape** — turns the flat vector into a 3D spatial volume (4×4×256)
3. **Four Conv2DTranspose blocks** — each doubles width and height: 4→8→16→32→64
4. **Output Conv2D** — produces the final 64×64×1 image with `tanh` activation

### Key design choices (from Brownlee Ch. 5 DCGAN best practices):
- `Conv2DTranspose` with stride=(2,2) and kernel=(4,4) — kernel is 2× the stride to avoid checkerboard artifacts
- `BatchNormalization` after each upsample layer — stabilizes training
- `LeakyReLU(0.2)` — better gradient flow than plain ReLU
- `tanh` output — outputs in [-1, 1] matching the normalized input images
- Generator is **not compiled** — it is never trained directly, only via the composite GAN model

In [ ]:
def build_generator(latent_dim):
    """
    DCGAN-style generator.
    Input  : random noise vector of shape (latent_dim,)
    Output : generated image of shape (64, 64, 1), values in [-1, 1]

    Upsampling path: 4×4 → 8×8 → 16×16 → 32×32 → 64×64
    """
    # Weight initializer recommended by Brownlee for stable GAN training
    init = RandomNormal(stddev=0.02)

    model = keras.Sequential(name='generator')

    # ── Foundation: Dense → Reshape into 4×4 spatial volume ───────────────
    # We need 256 feature maps each of size 4×4
    model.add(layers.Dense(256 * 4 * 4, input_dim=latent_dim,
                           kernel_initializer=init))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Reshape((4, 4, 256)))

    # ── Upsample Block 1: 4×4 → 8×8 ──────────────────────────────────────
    model.add(layers.Conv2DTranspose(128, (4, 4), strides=(2, 2),
                                     padding='same', kernel_initializer=init,
                                     use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # ── Upsample Block 2: 8×8 → 16×16 ────────────────────────────────────
    model.add(layers.Conv2DTranspose(128, (4, 4), strides=(2, 2),
                                     padding='same', kernel_initializer=init,
                                     use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # ── Upsample Block 3: 16×16 → 32×32 ──────────────────────────────────
    model.add(layers.Conv2DTranspose(64, (4, 4), strides=(2, 2),
                                     padding='same', kernel_initializer=init,
                                     use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # ── Upsample Block 4: 32×32 → 64×64 ──────────────────────────────────
    model.add(layers.Conv2DTranspose(32, (4, 4), strides=(2, 2),
                                     padding='same', kernel_initializer=init,
                                     use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # ── Output layer: 64×64×1 with tanh ──────────────────────────────────
    # tanh produces values in [-1, 1], matching our normalized training images
    model.add(layers.Conv2D(CHANNELS, (7, 7), activation='tanh',
                             padding='same', kernel_initializer=init))

    # Generator is NOT compiled — it trains only via the composite GAN model
    return model


# Quick check
gen_test = build_generator(LATENT_DIM)
gen_test.summary()
print(f"\nGenerator output shape: {gen_test.output_shape}")

---
## Step 5: Build the Discriminator

The discriminator takes a 64×64 grayscale image (real or fake) and outputs a single probability: how likely is this image to be real?

### Architecture (Conv2D downsampling → Dense → sigmoid):
1. Three Conv2D blocks, each halving spatial dimensions: 64→32→16→8
2. Flatten + Dropout for regularization
3. Dense(1, sigmoid) — binary output: real=1, fake=0

### Key design choices (DCGAN best practices):
- `Conv2D` with stride=(2,2) **instead of MaxPooling** — strided convolutions learn better downsampling
- `LeakyReLU(0.2)` — standard for discriminators; avoids dying ReLU problem
- `BatchNormalization` after first layer only (not the input or output layers)
- `Dropout(0.4)` — helps prevent the discriminator from becoming too strong and overwhelming the generator
- Adam with `lr=0.0002, beta_1=0.5` — the standard DCGAN optimizer settings
- Discriminator **is compiled** because it is trained directly on real/fake batches

In [ ]:
def build_discriminator(input_shape=(64, 64, 1)):
    """
    DCGAN-style discriminator.
    Input  : image of shape (64, 64, 1)
    Output : scalar probability in [0, 1] (1 = real, 0 = fake)

    Downsampling path: 64×64 → 32×32 → 16×16 → 8×8 → flatten → Dense
    """
    init = RandomNormal(stddev=0.02)

    model = keras.Sequential(name='discriminator')

    # ── Conv Block 1: 64×64 → 32×32 ───────────────────────────────────────
    # No BatchNorm on the first layer (DCGAN best practice)
    model.add(layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same',
                             kernel_initializer=init, input_shape=input_shape))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    # ── Conv Block 2: 32×32 → 16×16 ───────────────────────────────────────
    model.add(layers.Conv2D(128, (4, 4), strides=(2, 2), padding='same',
                             kernel_initializer=init))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    # ── Conv Block 3: 16×16 → 8×8 ─────────────────────────────────────────
    model.add(layers.Conv2D(256, (4, 4), strides=(2, 2), padding='same',
                             kernel_initializer=init))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    # ── Flatten + Output ───────────────────────────────────────────────────
    model.add(layers.Flatten())
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(1, activation='sigmoid',
                            kernel_initializer=init))

    # Compile: Adam with DCGAN standard learning rate and momentum
    model.compile(
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


disc_test = build_discriminator((IMG_SIZE, IMG_SIZE, CHANNELS))
disc_test.summary()

---
## Step 6: Build the Composite GAN Model

The composite model chains Generator → Discriminator into a single model. This is how the generator gets trained indirectly:

1. A noise vector enters the generator → a fake image is produced
2. The fake image enters the discriminator → a real/fake prediction comes out
3. We compute loss against the label `1` ("pretend this is real")
4. Gradients flow **backwards through the discriminator into the generator weights**

**Critical:** We freeze `discriminator.trainable = False` inside the composite model. This means the discriminator weights are NOT updated during generator training steps — only the generator weights update. As Brownlee explains, the discriminator is updated separately in standalone mode (Step 7).

In [ ]:
def build_gan(generator, discriminator):
    """
    Build the composite GAN model: Generator → Discriminator.

    The discriminator weights are FROZEN in this model so that only
    the generator weights update during composite model training steps.
    The discriminator still trains normally in standalone mode.
    """
    # Freeze discriminator inside the composite model
    discriminator.trainable = False

    model = keras.Sequential(name='gan_composite')
    model.add(generator)
    model.add(discriminator)

    # Same Adam settings as the discriminator
    model.compile(
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy'
    )
    return model


print("Composite GAN model structure:")
print("  Input → Generator → fake image → Discriminator → real/fake prob")
print("  During composite training: ONLY generator weights update")
print("  During standalone D training: ONLY discriminator weights update")

---
## Step 7: Helper functions for the training loop

The GAN training loop has three operations per batch:
1. Sample real images from the dataset (label = 1)
2. Generate fake images from noise (label = 0) → train discriminator
3. Generate new noise → train generator via composite model (label = 1, fooling D)

We also include a function to visualize generated images every `SAMPLE_INTERVAL` epochs so we can monitor training quality visually — the only reliable GAN evaluation method.

In [ ]:
def generate_real_samples(dataset, n_samples):
    """
    Randomly select n_samples real images from the dataset.
    Returns images with label=1 (real).
    """
    idx = np.random.randint(0, dataset.shape[0], n_samples)
    X   = dataset[idx]
    y   = np.ones((n_samples, 1))   # label: real = 1
    return X, y


def generate_latent_points(latent_dim, n_samples):
    """
    Sample random points from the standard normal distribution.
    These are the noise vectors fed into the generator.
    """
    return np.random.randn(n_samples, latent_dim)


def generate_fake_samples(generator, latent_dim, n_samples):
    """
    Use the generator to produce n_samples fake images.
    Returns images with label=0 (fake).
    """
    noise   = generate_latent_points(latent_dim, n_samples)
    X_fake  = generator.predict(noise, verbose=0)
    y_fake  = np.zeros((n_samples, 1))   # label: fake = 0
    return X_fake, y_fake


def save_sample_images(generator, latent_dim, epoch, class_name, save_dir, n=16):
    """
    Generate n sample images and save a grid to disk for visual inspection.
    This is our main tool for monitoring GAN quality during training.
    """
    noise     = generate_latent_points(latent_dim, n)
    gen_imgs  = generator.predict(noise, verbose=0)

    # Rescale from [-1, 1] back to [0, 1] for display
    gen_imgs  = (gen_imgs + 1.0) / 2.0

    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(gen_imgs[i, :, :, 0], cmap='gray')
        ax.axis('off')

    plt.suptitle(f'{class_name} — GAN epoch {epoch}', fontsize=11)
    plt.tight_layout()

    fname = os.path.join(save_dir, f'{class_name.lower()}_epoch_{epoch:04d}.png')
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    plt.close()


print("Helper functions defined.")

---
## Step 8: The training loop

This is the core GAN algorithm. Each epoch consists of multiple batches, and each batch runs three steps:

**Step A — Train Discriminator on real images:**  
Half a batch of real images → label=1 → `d_model.train_on_batch()`

**Step B — Train Discriminator on fake images:**  
Generator produces half a batch of fakes → label=0 → `d_model.train_on_batch()`

**Step C — Train Generator (via composite model):**  
New noise → composite model → label=1 ("fool discriminator") → `gan_model.train_on_batch()`  
Discriminator is frozen here → only generator weights update

### What to watch during training:
- **D loss should hover around 0.5–0.7** — if it collapses to 0 or spikes to 2+, training is unstable
- **G loss should decrease gradually** — if it stays very high, the generator is failing to fool D
- **Visual samples every `SAMPLE_INTERVAL` epochs** — the only true quality measure

In [ ]:
def train_gan(generator, discriminator, gan_model, real_images,
               latent_dim, epochs, batch_size, class_name, sample_dir):
    """
    Full GAN training loop.

    Args:
        generator     : untrained generator model
        discriminator : untrained discriminator model (compiled)
        gan_model     : composite model (compiled, discriminator frozen)
        real_images   : numpy array of real X-rays, shape (N, 64, 64, 1), range [-1,1]
        latent_dim    : size of generator noise input
        epochs        : number of training epochs
        batch_size    : samples per batch
        class_name    : label for logging and saved files
        sample_dir    : directory to save visual samples

    Returns:
        history dict with d_loss, d_acc, g_loss per epoch
    """
    half_batch  = batch_size // 2
    batches_per_epoch = real_images.shape[0] // batch_size
    history = {'d_loss': [], 'd_acc': [], 'g_loss': []}

    print(f"\nTraining GAN for class: '{class_name}'")
    print(f"  Real images   : {real_images.shape[0]}")
    print(f"  Batch size    : {batch_size} (half_batch = {half_batch})")
    print(f"  Epochs        : {epochs}")
    print(f"  Batches/epoch : {batches_per_epoch}")
    print("-" * 60)

    for epoch in range(1, epochs + 1):
        epoch_d_loss, epoch_d_acc, epoch_g_loss = [], [], []

        for _ in range(batches_per_epoch):

            # ── Step A: Train D on real images (label = 1) ────────────────
            X_real, y_real = generate_real_samples(real_images, half_batch)
            d_loss_real, d_acc_real = discriminator.train_on_batch(X_real, y_real)

            # ── Step B: Train D on fake images (label = 0) ────────────────
            X_fake, y_fake = generate_fake_samples(generator, latent_dim, half_batch)
            d_loss_fake, d_acc_fake = discriminator.train_on_batch(X_fake, y_fake)

            # Average discriminator loss and accuracy over real + fake
            d_loss = 0.5 * (d_loss_real + d_loss_fake)
            d_acc  = 0.5 * (d_acc_real  + d_acc_fake)

            # ── Step C: Train G via composite model (label = 1, fool D) ───
            # Discriminator is FROZEN here — only generator weights update
            noise  = generate_latent_points(latent_dim, batch_size)
            y_gan  = np.ones((batch_size, 1))   # we want D to think these are real
            g_loss = gan_model.train_on_batch(noise, y_gan)

            epoch_d_loss.append(d_loss)
            epoch_d_acc.append(d_acc)
            epoch_g_loss.append(g_loss)

        # Record epoch-level averages
        mean_d_loss = np.mean(epoch_d_loss)
        mean_d_acc  = np.mean(epoch_d_acc)
        mean_g_loss = np.mean(epoch_g_loss)
        history['d_loss'].append(mean_d_loss)
        history['d_acc'].append(mean_d_acc)
        history['g_loss'].append(mean_g_loss)

        # Log every 10 epochs
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:4d}/{epochs} | "
                  f"D loss: {mean_d_loss:.4f} | D acc: {mean_d_acc:.3f} | "
                  f"G loss: {mean_g_loss:.4f}")

        # Save visual sample grid every SAMPLE_INTERVAL epochs
        if epoch % SAMPLE_INTERVAL == 0:
            save_sample_images(generator, latent_dim, epoch,
                               class_name, sample_dir)
            print(f"  [Saved sample images at epoch {epoch}]")

    print(f"\nGAN training complete for '{class_name}'.")
    return history


print("Training loop defined.")

---
## Step 9: Train one GAN per minority class

We train a separate GAN for each minority class. This allows each GAN to specialise on the visual patterns of that disease rather than trying to learn all classes at once.

**Expected training time on Colab T4 GPU:** ~15–30 min per class at 200 epochs.  
For your final project submission increase `GAN_EPOCHS` to 500 for better image quality.

In [ ]:
trained_generators = {}   # store the trained generator for each class
gan_histories      = {}   # store training history for each class

for class_name in MINORITY_CLASSES:
    print(f"\n{'='*60}")
    print(f"Training GAN for: {class_name}")
    print(f"{'='*60}")

    # 1. Load real images for this class
    real_images = load_class_images(train_df, class_name, img_size=IMG_SIZE)

    # 2. Build fresh models for this class
    generator     = build_generator(LATENT_DIM)
    discriminator = build_discriminator((IMG_SIZE, IMG_SIZE, CHANNELS))
    gan_model     = build_gan(generator, discriminator)

    # Create class-specific sample directory
    sample_dir = os.path.join(RESULTS_DIR, f'gan_samples_{class_name.lower().replace(" ", "_")}')
    os.makedirs(sample_dir, exist_ok=True)

    # 3. Train
    history = train_gan(
        generator, discriminator, gan_model,
        real_images, LATENT_DIM, GAN_EPOCHS, BATCH_SIZE,
        class_name, sample_dir
    )

    # 4. Save the trained generator to Drive
    gen_path = os.path.join(GAN_DIR, f'generator_{class_name.lower().replace(" ", "_")}.keras')
    generator.save(gen_path)
    print(f"Generator saved to: {gen_path}")

    # 5. Store for later use
    trained_generators[class_name] = generator
    gan_histories[class_name]      = history

print("\nAll GANs trained.")

---
## Step 10: Plot GAN training losses

Healthy GAN training shows:
- **D loss** settling around 0.5–0.7 (it can't reliably tell real from fake)
- **G loss** decreasing over time as the generator improves
- If D loss collapses to near-zero → discriminator is too dominant → generator can't learn
- If G loss collapses to near-zero → mode collapse → generator is producing one repetitive image

In [ ]:
fig, axes = plt.subplots(1, len(MINORITY_CLASSES), figsize=(14, 5))
if len(MINORITY_CLASSES) == 1:
    axes = [axes]

for ax, class_name in zip(axes, MINORITY_CLASSES):
    hist = gan_histories[class_name]
    ep   = range(1, len(hist['d_loss']) + 1)

    ax.plot(ep, hist['d_loss'], label='Discriminator loss', color='#F44336')
    ax.plot(ep, hist['g_loss'], label='Generator loss',     color='#2196F3')
    ax.axhline(y=0.693, color='gray', linestyle=':', alpha=0.7,
               label='Ideal D loss (ln2 ≈ 0.693)')
    ax.set_title(f'GAN Loss — {class_name}', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('GAN Training Losses per Minority Class', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gan_training_losses.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Note: Ideal discriminator loss at convergence is ln(2) ≈ 0.693 — the point where D is at 50/50 chance.")

---
## Step 11: Visual quality check — final generated samples

Always inspect generated images before using them to augment a training set. Look for:
- Images that look like X-rays (grayscale chest structures visible)
- Variety between samples (no mode collapse — repeated identical patterns)
- Reasonable lung/chest shape and intensity distribution

In [ ]:
fig, all_axes = plt.subplots(len(MINORITY_CLASSES), 8, figsize=(16, 4 * len(MINORITY_CLASSES)))
if len(MINORITY_CLASSES) == 1:
    all_axes = [all_axes]

for row_axes, class_name in zip(all_axes, MINORITY_CLASSES):
    generator  = trained_generators[class_name]
    noise      = generate_latent_points(LATENT_DIM, 8)
    gen_imgs   = generator.predict(noise, verbose=0)
    gen_imgs   = (gen_imgs + 1.0) / 2.0   # rescale to [0, 1] for display

    for ax, img in zip(row_axes, gen_imgs):
        ax.imshow(img[:, :, 0], cmap='gray')
        ax.set_title(class_name, fontsize=8)
        ax.axis('off')

plt.suptitle('Final GAN Generated X-Ray Samples (after training)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gan_final_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12: Generate synthetic images and save to disk

Now that the generators are trained we use them to produce `N_SYNTHETIC_PER_CLASS` new images for each minority class. These get saved as PNG files and their paths + labels get added to a new augmented training CSV that Phase 4 will load.

**Note on resolution:** The GAN generates 64×64 images. We save them as-is — the Phase 4 preprocessing pipeline resizes everything to 224×224 anyway, so resolution upscaling happens at load time.

In [ ]:
import cv2

synthetic_records = []   # list of dicts to build the augmented CSV

for class_name in MINORITY_CLASSES:
    generator   = trained_generators[class_name]
    class_dir   = os.path.join(SYNTH_DIR, class_name.lower().replace(' ', '_'))
    os.makedirs(class_dir, exist_ok=True)

    print(f"Generating {N_SYNTHETIC_PER_CLASS} synthetic images for '{class_name}'...")

    # Generate in batches to avoid OOM
    generated = 0
    while generated < N_SYNTHETIC_PER_CLASS:
        n_batch = min(BATCH_SIZE, N_SYNTHETIC_PER_CLASS - generated)
        noise   = generate_latent_points(LATENT_DIM, n_batch)
        imgs    = generator.predict(noise, verbose=0)     # shape: (n, 64, 64, 1)

        for img in imgs:
            # Rescale from [-1, 1] → [0, 255] for saving as PNG
            img_uint8 = ((img + 1.0) / 2.0 * 255).astype(np.uint8).squeeze()
            fname = f'synthetic_{class_name.lower().replace(" ","_")}_{generated:05d}.png'
            fpath = os.path.join(class_dir, fname)
            cv2.imwrite(fpath, img_uint8)

            # Build label record — same format as Phase 1 train.csv
            record = {'filepath': fpath, 'filename': fname, 'is_synthetic': True}
            for cls in ALL_CLASSES:
                record[cls] = 1 if cls == class_name else 0
            synthetic_records.append(record)

            generated += 1
            if generated >= N_SYNTHETIC_PER_CLASS:
                break

    print(f"  Saved {generated} synthetic '{class_name}' images to {class_dir}")

synthetic_df = pd.DataFrame(synthetic_records)
print(f"\nTotal synthetic images generated: {len(synthetic_df)}")
print("Synthetic class distribution:")
print(synthetic_df[ALL_CLASSES].sum())

---
## Step 13: Build the augmented training CSV for Phase 4

We combine the original real training images with the synthetic ones into a single CSV. Phase 4 will load this CSV instead of the original `train.csv`.

The `is_synthetic` column lets us track which images are real vs generated — useful for ablation studies.

In [ ]:
# Add is_synthetic=False column to the real training data
real_train = train_df.copy()
real_train['is_synthetic'] = False

# Concatenate real + synthetic
augmented_train_df = pd.concat([real_train, synthetic_df], ignore_index=True)
augmented_train_df = augmented_train_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

# Save to Drive
aug_csv_path = os.path.join(SPLITS_DIR, 'train_augmented.csv')
augmented_train_df.to_csv(aug_csv_path, index=False)

print(f"Augmented training CSV saved to: {aug_csv_path}")
print(f"\nClass distribution — original train:")
print(real_train[ALL_CLASSES].sum().sort_values(ascending=False))
print(f"\nClass distribution — augmented train:")
print(augmented_train_df[ALL_CLASSES].sum().sort_values(ascending=False))
print(f"\nTotal images: {len(real_train):,} real + {len(synthetic_df):,} synthetic = {len(augmented_train_df):,}")

---
## Step 14: Plot before/after class distribution comparison

This plot is a key figure for your final report — it visually demonstrates the effect of the GAN augmentation on class balance.

In [ ]:
original_counts  = real_train[ALL_CLASSES].sum().sort_values(ascending=False)
augmented_counts = augmented_train_df[ALL_CLASSES].sum().sort_values(ascending=False)

x    = np.arange(len(ALL_CLASSES))
w    = 0.35
fig, ax = plt.subplots(figsize=(11, 5))

bars1 = ax.bar(x - w/2, original_counts[ALL_CLASSES], w,
               label='Original (real only)', color='#90CAF9', edgecolor='white')
bars2 = ax.bar(x + w/2, augmented_counts[ALL_CLASSES], w,
               label='Augmented (real + GAN)', color='#1565C0', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(ALL_CLASSES)
ax.set_ylabel('Number of images')
ax.set_title('Class distribution: before vs after GAN augmentation', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_distribution_before_after_gan.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 3 Summary

| Step | What we did | Output |
|------|-------------|--------|
| 1–2  | Setup + configuration | Environment ready |
| 3    | Loaded minority class images, normalized to [-1,1] | `real_images` arrays |
| 4    | Built DCGAN Generator (Dense→Reshape→4×Conv2DTranspose→tanh) | `generator` model |
| 5    | Built DCGAN Discriminator (3×Conv2D→Flatten→sigmoid) | `discriminator` model |
| 6    | Built composite GAN (G→D, D frozen for G updates) | `gan_model` |
| 7    | Defined helper functions (real/fake sampling, visualization) | Training utilities |
| 8    | Defined training loop (D on real, D on fake, G via composite) | `train_gan()` |
| 9    | Trained one GAN per minority class | Saved `.keras` generator files |
| 10   | Plotted D and G loss curves | `gan_training_losses.png` |
| 11   | Visual quality check on final generated images | `gan_final_samples.png` |
| 12   | Generated `N_SYNTHETIC_PER_CLASS` PNG images per class | Synthetic image files |
| 13   | Built `train_augmented.csv` (real + synthetic) | Ready for Phase 4 |
| 14   | Before/after class distribution chart | `class_distribution_before_after_gan.png` |

### Key things to note for your report:
- The discriminator loss should settle near **0.693** (ln(2)) at convergence — the mathematically ideal point where it can't reliably distinguish real from fake
- The before/after distribution chart is your core evidence that the GAN successfully addressed the class imbalance problem
- Visually check the generated X-rays — if they look like noise or repeated patterns, the GAN needs more training epochs or hyperparameter tuning

**Next step → Phase 4:** Retrain the CNN on `train_augmented.csv` and compare test metrics against the Phase 2 baseline.